# 07 · Análisis exploratorio: A-consigna con hiperparámetros propios

**Etapa 10 del plan.** A-consigna usa la ventana y las variables de la consigna, pero hereda los hiperparámetros de A, optimizados sobre 201905–06. Este notebook los optimiza sobre 201903–06, como pide la consigna, y compara:
- **`ACO`:** con hiperparámetros propios;
- **`AC`:** con los del `03` (control, ya corrido en el `06`).

**Todo lo de este notebook es exploratorio.** 201909 y 201910 ya se miraron, así que no demuestra nada ni cambia el resultado de la regla pre-registrada. La comparación y el criterio de lectura se fijaron en la ficha, antes de ver estos resultados.

**No escribe en `exp/03/`, `exp/05/`, `exp/06/` ni en los `resultados/03_*` a `06_*`:** solo los lee.

Ficha: `fichas/etapa_10_optuna_consigna.md` (en el laboratorio).

## 1. Parámetros

- **Retomar:** si la corrida se corta, se vuelve a correr entera. Las pruebas de Optuna y las semillas ya hechas no se repiten.
- **Corrida de prueba:** `N_PRUEBAS_OPTUNA = 2` y `N_SEMILLAS = 2`, en un directorio aparte.

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import platform
import time
import warnings

import duckdb
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats
from sklearn.metrics import roc_auc_score

# --- Parámetros -------------------------------------------------------------
BASE_PARQUET = Path('../data/base_2019.parquet')
RESULTADOS = Path('../resultados')
EXP_06 = Path('../exp/06')                          # solo se lee
EXP = Path('../exp/07')
FIGURAS = RESULTADOS / 'figuras'

CONTROL = 'AC'
CONFIG = 'ACO'

MESES_TRAIN = [201903, 201904, 201905, 201906]
MES_VALID = 201908
MESES_TEST = [201909, 201910]
MESES_EVAL = [MES_VALID] + MESES_TEST
ROTOS_EXTRA = [('ctarjeta_master_descuentos', 201910), ('mtarjeta_master_descuentos', 201910)]

SEMILLA_PRIMIGENIA = 979201
N_SEMILLAS = 10
N_PRUEBAS_OPTUNA = 50
ARBOLES_MAX = 3000
PACIENCIA = 100
N_BOOTSTRAP = 2_000
NIVEL_IC = 0.95
BANDA = (3_000, 6_000)                              # ambos incluidos
POSITIVOS = ['BAJA+1', 'BAJA+2']
POSITIVOS_AUC = ['BAJA+2']
N_HILOS = 8

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500
VENTANA_SUAVIZADO = 501
LARGO_CURVA = 25_000
# -----------------------------------------------------------------------------

PRED = EXP / 'pred' / CONFIG
PRED_CONTROL = EXP_06 / 'pred' / CONTROL
STORAGE = f"sqlite:///{(EXP / 'optuna.db').as_posix()}"
HIPER_JSON = RESULTADOS / '07_hiperparametros.json'
for d in (PRED, FIGURAS):
    d.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings('ignore', message='.*ChainedAssignmentError.*')  # falso positivo pandas 2.2 + Py 3.14
optuna.logging.set_verbosity(optuna.logging.WARNING)

ESPERADO_FILAS_POR_MES = {
    201901: 123_916, 201902: 125_020, 201903: 125_622, 201904: 126_116, 201905: 126_763,
    201906: 128_370, 201907: 129_853, 201908: 131_775, 201909: 133_419, 201910: 135_633,
}
ESPERADO_EVAL = {201908: (131_775, 543), 201909: (133_419, 615), 201910: (135_633, 632)}   # filas, BAJA+2
ESPERADO_COLUMNAS = 760
ESPERADO_ROTOS = 23


def primos_6_digitos():
    criba = np.ones(1_000_000, dtype=bool)
    criba[:2] = False
    for i in range(2, 1_001):
        if criba[i]:
            criba[i * i::i] = False
    return np.flatnonzero(criba[100_000:]) + 100_000


TODAS = np.random.default_rng(SEMILLA_PRIMIGENIA).choice(primos_6_digitos(), size=20, replace=False).tolist()[:10]
assert TODAS == [294103, 342929, 161093, 677561, 431077, 330679, 511579, 312773, 118147, 567529]
SEMILLAS = TODAS[:N_SEMILLAS]


def huella_archivos():
    """Control 6: huella de las salidas anteriores, para comprobar que no cambian."""
    return {p.name: hashlib.blake2b(p.read_bytes(), digest_size=16).hexdigest()
            for p in sorted(RESULTADOS.glob('0[3456]_*'))}


HUELLA_INICIAL = huella_archivos()
T_INICIO = time.time()
print('Salidas anteriores registradas:', len(HUELLA_INICIAL))

## 2. Funciones y carga

Las mismas definiciones del `03` al `06`, con los 25 casos de meses rotos a NA, igual que `AC`.

In [ ]:
def curva(prob, es_b2):
    orden = np.argsort(-prob, kind='stable')
    return np.cumsum(np.where(es_b2[orden], GANANCIA_ACIERTO, -COSTO_ESTIMULO))[:LARGO_CURVA].astype(np.float64)


def elegir_n(curva_media):
    suave = pd.Series(curva_media).rolling(VENTANA_SUAVIZADO, center=True, min_periods=VENTANA_SUAVIZADO).mean().to_numpy()
    return int(np.nanargmax(suave)) + 1


def banda(cv):
    return cv[..., BANDA[0] - 1:BANDA[1]].mean(axis=-1)


def wilcoxon_exacto(d):
    d = np.round(np.asarray(d, dtype=np.float64), 6)
    d = d[d != 0]
    n = len(d)
    if n == 0:
        return 1.0
    rangos = stats.rankdata(np.abs(d))
    w = float((np.sign(d) * rangos).sum())
    signos = 1 - 2 * ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1)
    return float(np.mean(np.abs(signos @ rangos) >= abs(w) - 1e-9))


def ic(x):
    x = np.asarray(x, dtype=np.float64)
    media = x.mean()
    medio = stats.t.ppf((1 + NIVEL_IC) / 2, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))
    return media, media - medio, media + medio


assert abs(wilcoxon_exacto(np.arange(1, 11)) - 2 / 1024) < 1e-12
assert abs(wilcoxon_exacto([1, 1, 1, 1, 1, 1, 1, 1, 1, -1]) - 22 / 1024) < 1e-12

HIPER_03 = json.loads((RESULTADOS / '03_hiperparametros.json').read_text())
semillas_06 = pd.read_csv(RESULTADOS / '06_semillas.csv')
HUELLA_06 = json.loads((EXP_06 / 'huellas.json').read_text())[CONTROL]

t0 = time.time()
con = duckdb.connect()
base = con.execute(f"""
    select * from read_parquet('{BASE_PARQUET.as_posix()}')
    where foto_mes <= {max(MESES_TEST)}
    order by foto_mes, numero_de_cliente
""").df()
con.close()
assert base['foto_mes'].value_counts().sort_index().to_dict() == ESPERADO_FILAS_POR_MES

CLAVES = ['numero_de_cliente', 'foto_mes', 'clase_ternaria']
VARIABLES = [c for c in base.columns if c not in CLAVES]
assert len(VARIABLES) == 152

rotos = pd.read_csv(RESULTADOS / '01_meses_rotos.csv')[['variable', 'foto_mes']]
assert len(rotos) == ESPERADO_ROTOS
rotos = pd.concat([rotos, pd.DataFrame(ROTOS_EXTRA, columns=['variable', 'foto_mes'])], ignore_index=True)
for v, mes in rotos.itertuples(index=False):
    base.loc[base['foto_mes'] == mes, v] = np.nan

FOTO_MES = base['foto_mes'].to_numpy()
PERIODO = (FOTO_MES // 100 * 12 + FOTO_MES % 100).astype(np.int64)
CLAVE = base['numero_de_cliente'].to_numpy(np.int64) * 100_000 + PERIODO
INDICE = pd.Index(CLAVE)
assert INDICE.is_unique
X_MES = np.asfortranarray(base[VARIABLES].to_numpy(np.float32))
CLASE = base['clase_ternaria'].to_numpy()
del base
gc.collect()
print(f'Dataset cargado en {time.time() - t0:.0f} s · {len(rotos)} casos de meses rotos a NA')

## 3. Tabla de variables

Las 760 columnas de A, como en el `03` y el `06`. **Control:** su huella tiene que ser la de `AC`.

In [ ]:
def construir():
    filas = np.flatnonzero(np.isin(FOTO_MES, MESES_TRAIN + MESES_EVAL))
    mes_fila = FOTO_MES[filas]
    n_train = int(np.isin(mes_fila, MESES_TRAIN).sum())
    assert np.isin(mes_fila[:n_train], MESES_TRAIN).all() and np.isin(mes_fila[n_train:], MESES_EVAL).all()
    pos = {k: INDICE.get_indexer(CLAVE[filas] - k) for k in range(3)}
    espec = [('original', v, 0) for v in VARIABLES]
    espec += [(op, v, k) for k in (1, 2) for op in ('lag', 'delta') for v in VARIABLES]
    ubic = {e: i for i, e in enumerate(espec)}
    F = np.empty((len(filas), len(espec)), dtype=np.float32)
    with np.errstate(invalid='ignore'):
        for j, v in enumerate(VARIABLES):
            F[:, ubic[('original', v, 0)]] = X_MES[filas, j]
            hk = {}
            for k in range(3):
                col = X_MES[pos[k], j]
                col[pos[k] < 0] = np.nan
                hk[k] = col
            for k in (1, 2):
                F[:, ubic[('lag', v, k)]] = hk[k]
                F[:, ubic[('delta', v, k)]] = hk[0] - hk[k]
    columnas = [v if op == 'original' else f'{v}_{op}{k}' for op, v, k in espec]
    return {'F': F, 'columnas': columnas, 'filas': filas, 'mes_fila': mes_fila, 'n_train': n_train}


def huella_control(T):
    hsh = hashlib.blake2b(digest_size=16)
    hsh.update(T['filas'].tobytes())
    for j in range(ESPERADO_COLUMNAS):
        hsh.update(T['F'][:, j].tobytes())
    return hsh.hexdigest()


T = construir()
assert T['F'].shape[1] == ESPERADO_COLUMNAS
assert huella_control(T) == HUELLA_06, 'Las columnas no son las de AC'     # control 2
EVALUA = slice(T['n_train'], None)
MES = T['mes_fila'][EVALUA]
clase_eval = CLASE[T['filas'][EVALUA]]
ES_B2 = clase_eval == 'BAJA+2'
es_mes = {m: MES == m for m in MESES_EVAL}
for mes, (filas, b2) in ESPERADO_EVAL.items():            # control 1
    assert (es_mes[mes].sum(), ES_B2[es_mes[mes]].sum()) == (filas, b2), mes
print(f'{T["n_train"]:,} filas de entrenamiento · {T["F"].shape[1]} variables')

## 4. Optuna sobre 201903–06

La misma búsqueda del `03`: los 5 hiperparámetros de la consigna, con el mismo rango, AUC en 201908 contra `BAJA+2` y *early stopping*. La prueba 0 son los hiperparámetros del `03`, así que la búsqueda no puede terminar peor en validación.

**Si ya existe `resultados/07_hiperparametros.json`, no se corre.**

In [ ]:
FIJOS = {'objective': 'binary', 'metric': 'auc', 'boosting': 'gbdt', 'max_bin': 31, 'bagging_freq': 1,
         'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_gain_to_split': 0.0, 'feature_pre_filter': False,
         'num_threads': N_HILOS, 'deterministic': True, 'force_col_wise': True, 'verbose': -1}

n = T['n_train']
y_train = np.isin(CLASE[T['filas'][:n]], POSITIVOS).astype(np.int8)
dtrain = lgb.Dataset(T['F'][:n], label=y_train, feature_name=T['columnas'],
                     params={'max_bin': 31, 'feature_pre_filter': False}, free_raw_data=False)

if HIPER_JSON.exists():
    HIPER = json.loads(HIPER_JSON.read_text())
    print(f'Se usan los hiperparámetros guardados: prueba {HIPER["prueba"]}, AUC {HIPER["auc_201908"]:.5f}')
else:
    t0 = time.time()
    es_valid = es_mes[MES_VALID]
    dvalid = lgb.Dataset(T['F'][EVALUA][es_valid], label=np.isin(clase_eval[es_valid], POSITIVOS_AUC).astype(np.int8),
                         reference=dtrain, free_raw_data=False)

    def objetivo(trial):
        p = {**FIJOS,
             'num_leaves': trial.suggest_int('num_leaves', 8, 512, log=True),
             'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 2000, log=True),
             'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
             'feature_fraction': trial.suggest_float('feature_fraction', 0.2, 1.0),
             'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
             'seed': TODAS[0]}
        t1 = time.time()
        modelo = lgb.train(p, dtrain, num_boost_round=ARBOLES_MAX, valid_sets=[dvalid], valid_names=['valid'],
                           callbacks=[lgb.early_stopping(PACIENCIA, verbose=False)])
        auc = modelo.best_score['valid']['auc']
        trial.set_user_attr('arboles', modelo.best_iteration)
        print(f'prueba {trial.number}: AUC {auc:.5f} · {modelo.best_iteration} árboles · hojas {p["num_leaves"]} · '
              f'mdl {p["min_data_in_leaf"]} · lr {p["learning_rate"]:.3f} · ff {p["feature_fraction"]:.2f} · '
              f'bf {p["bagging_fraction"]:.2f} · {time.time() - t1:.0f} s')
        return auc

    estudio = optuna.create_study(study_name='07_ACO', storage=STORAGE, direction='maximize', load_if_exists=True,
                                  sampler=optuna.samplers.TPESampler(seed=SEMILLA_PRIMIGENIA))
    if len(estudio.trials) == 0:
        estudio.enqueue_trial(HIPER_03['params'])
    hechas = sum(t.state == optuna.trial.TrialState.COMPLETE for t in estudio.trials)
    estudio.optimize(objetivo, n_trials=max(0, N_PRUEBAS_OPTUNA - hechas))
    completas = [t for t in estudio.trials if t.state == optuna.trial.TrialState.COMPLETE]
    HIPER = {'prueba': estudio.best_trial.number, 'params': estudio.best_trial.params,
             'arboles': estudio.best_trial.user_attrs['arboles'], 'auc_201908': estudio.best_value,
             'pruebas': len(completas), 'auc_prueba_0': completas[0].value, 'arboles_prueba_0': completas[0].user_attrs['arboles'],
             'maquina': platform.node(), 'fecha_hora': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}
    HIPER_JSON.write_text(json.dumps(HIPER, indent=2))
    del dvalid
    gc.collect()
    print(f'Optuna: {(time.time() - t0) / 60:.1f} min')

print({k: round(v, 4) if isinstance(v, float) else v for k, v in HIPER['params'].items()}, f'· {HIPER["arboles"]} árboles')
print(f'Del 03: {HIPER_03["params"]} · {HIPER_03["arboles"]} árboles')
print(f'AUC 201908: mejor prueba {HIPER["auc_201908"]:.5f} (prueba {HIPER["prueba"]}) · prueba 0, con los del 03: {HIPER["auc_prueba_0"]:.5f}')

## 5. Modelos

**Control:** `ACO` con la primera semilla tiene que dar la AUC de la mejor prueba de Optuna.

In [ ]:
MEJORES = {**FIJOS, **HIPER['params']}
segundos = []
for semilla in SEMILLAS:
    archivo = PRED / f'{semilla}.npy'
    if archivo.exists():
        continue
    t0 = time.time()
    modelo = lgb.train({**MEJORES, 'seed': semilla}, dtrain, num_boost_round=HIPER['arboles'])
    prob = modelo.predict(T['F'][EVALUA], num_threads=N_HILOS).astype(np.float32)
    if semilla == TODAS[0]:                               # control 3
        auc = roc_auc_score(ES_B2[es_mes[MES_VALID]], prob[es_mes[MES_VALID]])
        print(f'Reproducción: AUC {auc:.6f} · la de Optuna: {HIPER["auc_201908"]:.6f}')
        if HIPER['maquina'] == platform.node():
            assert abs(auc - HIPER['auc_201908']) < 1e-6
    np.save(archivo, prob)
    segundos.append(time.time() - t0)
    print(f'{CONFIG} · semilla {semilla}: {time.time() - t0:.0f} s')
del dtrain
gc.collect()

faltan = [s for s in SEMILLAS if not (PRED / f'{s}.npy').exists()]
assert not faltan, faltan                                 # control 8
PROB = {CONFIG: np.stack([np.load(PRED / f'{s}.npy') for s in SEMILLAS]),
        CONTROL: np.stack([np.load(PRED_CONTROL / f'{s}.npy') for s in SEMILLAS])}

## 6. Comparación

`ACO` contra `AC`, en 201909 y 201910:
- **Por semilla:** N en la meseta de la curva media de 201908; IC 95 % con t de Student; Wilcoxon exacto (p descriptivo); banda.
- **Ensamble:** promedio de las probabilidades de las semillas, N elegido en 201908, bootstrap pareado de clientes.

In [ ]:
curvas = {(c, m): np.stack([curva(p[es_mes[m]], ES_B2[es_mes[m]]) for p in PROB[c]]) for c in PROB for m in MESES_EVAL}
N = {c: elegir_n(curvas[(c, MES_VALID)].mean(axis=0)) for c in PROB}
G = {k: v[:, N[k[0]] - 1] / 1e6 for k, v in curvas.items()}
B = {k: banda(v) / 1e6 for k, v in curvas.items()}

t06 = semillas_06[(semillas_06['config'] == CONTROL) & semillas_06['semilla'].isin(SEMILLAS)]     # control 5
assert (t06['N'] == N[CONTROL]).all(), (t06['N'].unique(), N[CONTROL])
if N_SEMILLAS == 10:
    assert N[CONTROL] == 3_793
for m in MESES_EVAL:
    esperado = t06[t06['foto_mes'] == m].set_index('semilla').loc[SEMILLAS, 'ganancia_M'].to_numpy()
    np.testing.assert_allclose(G[(CONTROL, m)], esperado, atol=6e-5)

filas = []
for c in PROB:
    for m in MESES_EVAL:
        for i, s in enumerate(SEMILLAS):
            filas.append({'config': c, 'semilla': s, 'foto_mes': m, 'N': N[c], 'ganancia_M': round(G[(c, m)][i], 4),
                          'ganancia_banda_M': round(B[(c, m)][i], 4),
                          'auc': round(roc_auc_score(ES_B2[es_mes[m]], PROB[c][i][es_mes[m]]), 6)})
pd.DataFrame(filas).to_csv(RESULTADOS / '07_semillas.csv', index=False)

ENSAMBLE = {c: PROB[c].astype(np.float64).mean(axis=0) for c in PROB}
CURVA_E = {(c, m): curva(ENSAMBLE[c][es_mes[m]], ES_B2[es_mes[m]]) for c in PROB for m in MESES_EVAL}
N_E = {c: elegir_n(CURVA_E[(c, MES_VALID)]) for c in PROB}


def aporte(prob, es_b2, n):
    """Lo que deja cada cliente del mes: 0 si no se lo contacta."""
    contactado = np.zeros(len(prob), dtype=bool)
    contactado[np.argsort(-prob, kind='stable')[:n]] = True
    return np.where(contactado, np.where(es_b2, GANANCIA_ACIERTO, -COSTO_ESTIMULO), 0).astype(np.float64)


def remuestrear(d, n_veces, semilla):
    rng = np.random.default_rng(semilla)
    n = len(d)
    con_valor = np.flatnonzero(d)
    salida = np.empty(n_veces)
    for b in range(n_veces):
        veces = np.bincount(rng.integers(0, n, size=n), minlength=n)
        salida[b] = veces[con_valor] @ d[con_valor]
    return salida


filas = []
for m in MESES_TEST:
    d = (G[(CONFIG, m)] - G[(CONTROL, m)]).round(6)
    media, inf, sup = ic(d)
    filas.append({'medida': 'semillas', 'config': CONFIG, 'contra': CONTROL, 'foto_mes': m, 'N': N[CONFIG], 'N_contra': N[CONTROL],
                  'd_M': media, 'ic95_inf_M': inf, 'ic95_sup_M': sup, 'a_favor': f'{int((d > 0).sum())}/{len(d)}',
                  'p_descriptivo': wilcoxon_exacto(d), 'd_banda_M': (B[(CONFIG, m)] - B[(CONTROL, m)]).mean()})
    b2 = ES_B2[es_mes[m]]
    dif = aporte(ENSAMBLE[CONFIG][es_mes[m]], b2, N_E[CONFIG]) - aporte(ENSAMBLE[CONTROL][es_mes[m]], b2, N_E[CONTROL])
    sumas = remuestrear(dif, N_BOOTSTRAP, SEMILLA_PRIMIGENIA + m) / 1e6
    filas.append({'medida': 'ensamble', 'config': CONFIG, 'contra': CONTROL, 'foto_mes': m, 'N': N_E[CONFIG], 'N_contra': N_E[CONTROL],
                  'd_M': dif.sum() / 1e6, 'ic95_inf_M': np.percentile(sumas, 2.5), 'ic95_sup_M': np.percentile(sumas, 97.5),
                  'a_favor': f'{(sumas > 0).mean() * 100:.1f} %', 'p_descriptivo': np.nan,
                  'd_banda_M': (banda(CURVA_E[(CONFIG, m)]) - banda(CURVA_E[(CONTROL, m)])) / 1e6})
comparacion = pd.DataFrame(filas)
comparacion.round(4).to_csv(RESULTADOS / '07_comparacion.csv', index=False)
print('EXPLORATORIO')
print(comparacion.round(3).to_string(index=False))
print()
print(pd.DataFrame({c: {m: round(G[(c, m)].mean(), 2) for m in MESES_EVAL} for c in PROB}).to_string())

## 7. Figura, criterio de lectura y cierre

Los hiperparámetros propios **mejoran** a A-consigna si la diferencia es positiva en los dos meses en las tres medidas: media por semilla, banda por semilla y banda del ensamble (fijado antes de correr). No es una prueba estadística.

**Control:** las salidas anteriores no cambiaron.

In [ ]:
TINTA_2, GRILLA, AZUL = '#52514e', '#e1e0d9', '#2a78d6'
plt.rcParams.update({'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb', 'axes.edgecolor': '#c3c2b7',
                     'axes.labelcolor': TINTA_2, 'xtick.color': TINTA_2, 'ytick.color': TINTA_2, 'text.color': '#0b0b0b',
                     'axes.grid': True, 'grid.color': GRILLA, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 13, 'axes.titlesize': 15})


def coma(x, decimales=1):
    return f'{x:,.{decimales}f}'.replace(',', '_').replace('.', ',').replace('_', '.')


fig, ejes = plt.subplots(1, 2, figsize=(12.8, 7.2), sharex=True)
etiquetas = ['por semilla', 'ensamble']
y = np.array([1, 0])
for ax, m in zip(ejes, MESES_TEST):
    t = comparacion[comparacion['foto_mes'] == m].set_index('medida').loc[['semillas', 'ensamble']]
    ax.axvline(0, color=TINTA_2, lw=1.2)
    ax.hlines(y, t['ic95_inf_M'], t['ic95_sup_M'], color=AZUL, lw=2)
    ax.scatter(t['d_M'], y, s=70, color=AZUL, zorder=3, edgecolor='#fcfcfb')
    ax.set_yticks(y, etiquetas)
    ax.set_ylim(-0.7, 1.7)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: coma(v, 0 if float(v).is_integer() else 1)))
    ax.set_title(str(m), loc='left')
    ax.set_xlabel('Diferencia contra AC (M$), IC 95 %')
fig.suptitle('Exploratorio: A-consigna con hiperparámetros propios contra prestados', x=0.02, ha='left', fontsize=17)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURAS / '07_1_hiperparametros_propios.png', dpi=150)
plt.show()

assert huella_archivos() == HUELLA_INICIAL, 'Cambiaron las salidas anteriores'   # control 6
print(f'Control de salidas OK · {(time.time() - T_INICIO) / 60:.1f} min en total')
if segundos:
    print(f'Segundos por modelo: {np.mean(segundos):.1f}')
print()
print('EXPLORATORIO: no cambia el resultado de la regla pre-registrada.')
s = comparacion[comparacion['medida'] == 'semillas']
e = comparacion[comparacion['medida'] == 'ensamble']
uno, dos, tres = (s['d_M'] > 0).all(), (s['d_banda_M'] > 0).all(), (e['d_banda_M'] > 0).all()
print(f'Media por semilla {"sí" if uno else "no"} · banda por semilla {"sí" if dos else "no"} · banda del ensamble {"sí" if tres else "no"}')
print('Los hiperparámetros propios mejoran a A-consigna:', 'sí' if uno and dos and tres else 'no')